# Exchange Rates Pipeline (Databricks)

## Descripción general

Este proyecto implementa un pipeline de datos en Databricks para procesar información de tipos de cambio (exchange rates) desde una API externa.

El flujo sigue una arquitectura tipo Medallion (Bronze, Silver, Gold) para organizar los datos de forma escalable y fácil de mantener e idempotente.





In [0]:
%sql
CREATE CATALOG IF NOT EXISTS finance;
CREATE SCHEMA IF NOT EXISTS finance.exchange_rates;

USE CATALOG finance;
USE SCHEMA exchange_rates;

In [0]:
import requests

# aquí traigo datos de la API de Frankfurter para un rango de fechas
# uso USD como base y algunas monedas específicas
def extract_data():
    url = "https://api.frankfurter.dev/v1/2024-01-01..2024-01-10?base=USD&symbols=MXN,EUR,BRL,COP"
    
    response = requests.get(url, timeout=10)
    data = response.json()

    rows = []

    # recorro la respuesta para convertirla a filas
    for date, rates in data["rates"].items():
        for currency, rate in rates.items():
            rows.append((date, currency, float(rate)))

    return rows


# creo dataframe y lo guardo como bronze (datos crudos)
rows = extract_data()
df = spark.createDataFrame(rows, ["date", "currency", "rate"])

df.write.format("delta").mode("overwrite").saveAsTable(
    "finance.exchange_rates.currency_bronze"
)
print("bronze listo")


In [0]:
%sql
SELECT * FROM finance.exchange_rates.currency_bronze LIMIT 10;

In [0]:
# [TABLE_OR_VIEW_NOT_FOUND] The table or view `finance`.`exchange_rates`.`currency_bronze` cannot be found. Verify the spelling and correctness of the schema and catalog.

from pyspark.sql.functions import col, to_date, year, month, current_timestamp, lit


spark.sql("USE CATALOG finance")
spark.sql("USE SCHEMA exchange_rates")

# leo datos crudos
df = spark.table("currency_bronze")


# limpio datos y agrego columnas útiles
df_silver = df \
    .filter(col("rate") > 0) \
    .withColumn("date", to_date(col("date"))) \
    .withColumn("year", year(col("date"))) \
    .withColumn("month", month(col("date"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("updated_at", current_timestamp()) \
    .withColumn("operation_type", lit("INSERT"))


spark.sql("""
CREATE OR REPLACE TABLE finance.exchange_rates.currency_silver (
    date DATE,
    currency STRING,
    rate DOUBLE,
    year INT,
    month INT,
    ingestion_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (year, month)
""")


df_silver.createOrReplaceTempView("tmp_silver")

spark.sql("""
MERGE INTO finance.exchange_rates.currency_silver AS A
USING tmp_silver AS B
ON A.date = B.date AND A.currency = B.currency

WHEN MATCHED AND A.rate != B.rate THEN
    UPDATE SET
        A.rate = B.rate,
        A.year = B.year,
        A.month = B.month,
        A.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
    INSERT (
        date, currency, rate, year, month,
        ingestion_timestamp, updated_at
    )
    VALUES (
        B.date, B.currency, B.rate, B.year, B.month,
        B.ingestion_timestamp, B.updated_at
    )
""")

print("silver listo")

In [0]:
%sql
SELECT * FROM finance.exchange_rates.currency_silver LIMIT 10;

In [0]:
from pyspark.sql.functions import current_timestamp, concat_ws, col

df_silver = spark.table("finance.exchange_rates.currency_silver")

# genero eventos a partir de los datos ya procesados
df_events = df_silver \
    .withColumn("event_timestamp", current_timestamp()) \
    .withColumn("entity_id", concat_ws("_", col("date"), col("currency"))) \
    .withColumn("event_type", lit("INSERT"))  # se agrega el tipo de cambio de registro



df_events = df_events.select(
    "event_type",
    "event_timestamp",
    "entity_id",
    "date",
    "currency",
    "rate"
)

# guardo eventos (tipo simulación de streaming)
df_events.write.mode("append").saveAsTable(
    "finance.exchange_rates.currency_events"
)

print("eventos listos")

In [0]:
%sql
SELECT * FROM finance.exchange_rates.currency_events LIMIT 10

In [0]:
from pyspark.sql.functions import avg, min, max, stddev, col, current_date, last_day

# leo silver (ya limpio y con lógica de negocio)
df_silver = spark.table("finance.exchange_rates.currency_silver")


# calculo métricas mensuales
df_stats = df_silver.groupBy("currency", "year", "month").agg(
    avg("rate").alias("avg_rate"),
    min("rate").alias("min_rate"),
    max("rate").alias("max_rate"),
    stddev("rate").alias("volatility")
)


# uno métricas con el detalle original
df_gold_new = df_silver.join(
    df_stats,
    on=["currency", "year", "month"],
    how="left"
)


# marco anomalías (arriba o abajo del rango esperado)
df_gold_new = df_gold_new.withColumn(
    "is_anomaly",
    (col("rate") > (col("avg_rate") + 2 * col("volatility"))) |
    (col("rate") < (col("avg_rate") - 2 * col("volatility")))
)


# agrego control de fechas
df_gold_new = df_gold_new \
    .withColumn("fechacarga", current_date()) \
    .withColumn("fechacorte", last_day(col("date")))  # fin de mes


# creo tabla si no existe
spark.sql("""
CREATE TABLE IF NOT EXISTS finance.exchange_rates.currency_gold (
    date DATE,
    currency STRING,
    rate DOUBLE,
    year INT,
    month INT,
    avg_rate DOUBLE,
    min_rate DOUBLE,
    max_rate DOUBLE,
    volatility DOUBLE,
    is_anomaly BOOLEAN,
    fechacarga DATE,
    fechacorte DATE
)
USING DELTA
PARTITIONED BY (year, month)
""")


# inserto solo registros nuevos (evito duplicados)
df_gold_new.createOrReplaceTempView("tmp_gold")

spark.sql("""
MERGE INTO finance.exchange_rates.currency_gold AS A
USING tmp_gold AS B
ON A.date = B.date
AND A.currency = B.currency

WHEN NOT MATCHED THEN
  INSERT (
    date,
    currency,
    rate,
    year,
    month,
    avg_rate,
    min_rate,
    max_rate,
    volatility,
    is_anomaly,
    fechacarga,
    fechacorte
  )
  VALUES (
    B.date,
    B.currency,
    B.rate,
    B.year,
    B.month,
    B.avg_rate,
    B.min_rate,
    B.max_rate,
    B.volatility,
    B.is_anomaly,
    B.fechacarga,
    B.fechacorte
  )
""")

print("gold listo")

In [0]:
%sql
SELECT * FROM finance.exchange_rates.currency_gold

In [0]:
%sql
SELECT DATE(fechacorte),COUNT(*) AS TOTAL
FROM finance.exchange_rates.currency_gold
GROUP BY 1 ORDER BY 1 DESC;